## 2.1 理论计算题

给定字符序列 "ababc"，采用一阶马尔可夫模型 $p(x_t|x_{t-1})$，使用拉普拉斯平滑（加 1 平滑）估计条件概率。

词汇表为 {'a', 'b', 'c'}，计算时考虑所有可能转移，包括未出现的情况。

In [2]:
sequence = "ababc"
vocab = {'a', 'b', 'c'}
V = len(vocab)

print(f"字符序列: {sequence}")
print(f"词汇表: {vocab}")
print(f"词汇表大小 V: {V}")

字符序列: ababc
词汇表: {'b', 'c', 'a'}
词汇表大小 V: 3


In [3]:
transitions = {}
for char in vocab:
    transitions[char] = {c: 0 for c in vocab}

for i in range(1, len(sequence)):
    prev_char = sequence[i-1]
    curr_char = sequence[i]
    transitions[prev_char][curr_char] += 1

print("转移频率统计:")
for prev in sorted(vocab):
    for curr in sorted(vocab):
        print(f"  {prev} -> {curr}: {transitions[prev][curr]}")

转移频率统计:
  a -> a: 0
  a -> b: 2
  a -> c: 0
  b -> a: 1
  b -> b: 0
  b -> c: 1
  c -> a: 0
  c -> b: 0
  c -> c: 0


### 计算 $p(a | b)$

根据拉普拉斯平滑公式：

$p(x_t|x_{t-1}) = \frac{count(x_{t-1}, x_t) + 1}{count(x_{t-1}) + V}$

其中：
- $count(b, a)$：序列中 'b' 后面跟着 'a' 的次数
- $count(b)$：序列中 'b' 作为前一个字符出现的次数
- $V$：词汇表大小 = 3

In [4]:
count_b_a = transitions['b']['a']
count_b = sum(transitions['b'].values())

p_a_given_b = (count_b_a + 1) / (count_b + V)

print(f"count(b, a) = {count_b_a}")
print(f"count(b) = {count_b}")
print(f"p(a | b) = ({count_b_a} + 1) / ({count_b} + {V}) = {p_a_given_b}")

count(b, a) = 1
count(b) = 2
p(a | b) = (1 + 1) / (2 + 3) = 0.4


### 计算 $p(c | b)$

同样使用拉普拉斯平滑公式。

In [5]:
count_b_c = transitions['b']['c']

p_c_given_b = (count_b_c + 1) / (count_b + V)

print(f"count(b, c) = {count_b_c}")
print(f"count(b) = {count_b}")
print(f"p(c | b) = ({count_b_c} + 1) / ({count_b} + {V}) = {p_c_given_b}")

count(b, c) = 1
count(b) = 2
p(c | b) = (1 + 1) / (2 + 3) = 0.4


In [6]:
print("="*50)
print("理论计算题答案")
print("="*50)
print(f"1. p(a | b) = {p_a_given_b}")
print(f"2. p(c | b) = {p_c_given_b}")
print("="*50)

理论计算题答案
1. p(a | b) = 0.4
2. p(c | b) = 0.4


---

## 2.2 编程题



In [7]:
import re
from collections import Counter

def preprocess_text(text, n):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    word_counts = Counter(tokens)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, count) in enumerate(sorted_words)}
    features = []
    labels = []
    for i in range(len(tokens) - n + 1):
        feature = tokens[i:i+n]
        if i + n < len(tokens):
            label = tokens[i+n]
        else:
            label = None
        features.append(feature)
        labels.append(label)
    return vocab, (features, labels)

### 测试示例

In [8]:
text = "The time machine"
n = 2

vocab, (features, labels) = preprocess_text(text, n)

print("测试结果：")
print(f"输入文本: '{text}'")
print(f"n = {n}")
print(f"词汇表: {vocab}")
print(f"特征列表: {features}")
print(f"标签列表: {labels}")

expected_features = [['the', 'time'], ['time', 'machine']]
expected_labels = ['machine', None]

print(f"\n期望特征: {expected_features}")
print(f"期望标签: {expected_labels}")
print(f"特征匹配: {features == expected_features}")
print(f"标签匹配: {labels == expected_labels}")

测试结果：
输入文本: 'The time machine'
n = 2
词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time'], ['time', 'machine']]
标签列表: ['machine', None]

期望特征: [['the', 'time'], ['time', 'machine']]
期望标签: ['machine', None]
特征匹配: True
标签匹配: True


### 更多测试用例

In [9]:
text2 = "Hello, World! This is a test."
n2 = 3

vocab2, (features2, labels2) = preprocess_text(text2, n2)

print("\n测试用例 2：")
print(f"输入文本: '{text2}'")
print(f"n = {n2}")
print(f"词汇表: {vocab2}")
print(f"特征列表: {features2}")
print(f"标签列表: {labels2}")

text3 = "I love love love Python"
n3 = 2

vocab3, (features3, labels3) = preprocess_text(text3, n3)

print("\n测试用例 3：")
print(f"输入文本: '{text3}'")
print(f"n = {n3}")
print(f"词汇表: {vocab3}")
print(f"特征列表: {features3}")
print(f"标签列表: {labels3}")


测试用例 2：
输入文本: 'Hello, World! This is a test.'
n = 3
词汇表: {'a': 0, 'hello': 1, 'is': 2, 'test': 3, 'this': 4, 'world': 5}
特征列表: [['hello', 'world', 'this'], ['world', 'this', 'is'], ['this', 'is', 'a'], ['is', 'a', 'test']]
标签列表: ['is', 'a', 'test', None]

测试用例 3：
输入文本: 'I love love love Python'
n = 2
词汇表: {'love': 0, 'i': 1, 'python': 2}
特征列表: [['i', 'love'], ['love', 'love'], ['love', 'love'], ['love', 'python']]
标签列表: ['love', 'love', 'python', None]


## 3.1 理论计算题

考虑一个线性 RNN（无偏置），定义为：

$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$

输出：

$$o_t = W_{oh} h_t$$

假设损失函数为平方损失：

$$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$$

推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

### 梯度推导过程

**步骤 1：损失对输出的梯度**

$$\frac{\partial L}{\partial o_t} = o_t - y_t$$

**步骤 2：损失对隐藏状态的梯度**

$$\frac{\partial L}{\partial h_t} = \frac{\partial L}{\partial o_t} \cdot \frac{\partial o_t}{\partial h_t} = (o_t - y_t) W_{oh}^T$$

**步骤 3：通过时间反向传播**

$$\frac{\partial L}{\partial h_{t-1}} = \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial h_{t-1}} = \frac{\partial L}{\partial h_t} \cdot W_{hh}^T$$

递推可得：

$$\frac{\partial L}{\partial h_k} = \sum_{t=k+1}^T \frac{\partial L}{\partial h_t} \cdot (W_{hh}^T)^{t-k}$$

**步骤 4：损失对 $W_{hh}$ 的梯度**

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} \cdot h_{t-1}^T$$

将 $\frac{\partial L}{\partial h_t}$ 展开：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \left( \sum_{k=t}^T (o_k - y_k) W_{oh}^T (W_{hh}^T)^{k-t} \right) h_{t-1}^T$$


### 梯度消失与爆炸条件

梯度表达式中包含 $(W_{hh}^T)^{k-t}$ 项，这是一个矩阵幂。其行为取决于 $W_{hh}$ 的特征值：

1. **梯度消失**：当 $W_{hh}$ 的所有特征值的绝对值都小于 1 时，$(W_{hh}^T)^{k-t}$ 会随着 $k-t$ 的增大而趋近于 0。此时，远距离的梯度贡献会逐渐消失，导致模型难以学习长期依赖关系。

2. **梯度爆炸**：当 $W_{hh}$ 的最大特征值的绝对值大于 1 时，$(W_{hh}^T)^{k-t}$ 会随着 $k-t$ 的增大而指数增长。此时，梯度会变得非常大，导致训练不稳定。

**结论**：线性 RNN 无法处理长期依赖问题，因为梯度要么消失要么爆炸。这也是引入 LSTM 和 GRU 等门控机制的原因。

## 3.2 编程题




In [10]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单步前向传播
    
    参数：
    x_t: 形状 (batch_size, input_size)
    h_prev: 形状 (batch_size, hidden_size)
    W_hx: 形状 (hidden_size, input_size)
    W_hh: 形状 (hidden_size, hidden_size)
    b_h: 形状 (hidden_size,)
    
    返回：
    h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
    cache: 缓存用于反向传播
    """
    hidden_size = W_hh.shape[0]
    batch_size = x_t.shape[0]
    
    pre_act = np.dot(h_prev, W_hh.T) + np.dot(x_t, W_hx.T) + b_h
    h_t = np.tanh(pre_act)
    
    cache = (x_t, h_prev, W_hx, W_hh, b_h, h_t)
    
    return h_t, cache

In [11]:
def rnn_backward(dh_next, cache):
    """
    RNN 单步反向传播
    
    参数：
    dh_next: 上游梯度，损失对 h_t 的梯度，形状 (batch_size, hidden_size)
    cache: 前向传播缓存
    
    返回：
    dx_t: 输入梯度，形状 (batch_size, input_size)
    dh_prev: 前一隐藏状态梯度，形状 (batch_size, hidden_size)
    dW_hx: 权重梯度，形状 (hidden_size, input_size)
    dW_hh: 权重梯度，形状 (hidden_size, hidden_size)
    db_h: 偏置梯度，形状 (hidden_size,)
    """
    x_t, h_prev, W_hx, W_hh, b_h, h_t = cache
    batch_size = x_t.shape[0]
    
    dtanh = dh_next * (1 - h_t ** 2)
    
    dW_hx = np.dot(dtanh.T, x_t)
    dW_hh = np.dot(dtanh.T, h_prev)
    db_h = np.sum(dtanh, axis=0)
    
    dx_t = np.dot(dtanh, W_hx)
    dh_prev = np.dot(dtanh, W_hh)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

### 测试示例

In [12]:
np.random.seed(42)

batch_size = 2
input_size = 3
hidden_size = 4

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size,)

print("输入形状：")
print(f"x_t: {x_t.shape}")
print(f"h_prev: {h_prev.shape}")
print(f"W_hx: {W_hx.shape}")
print(f"W_hh: {W_hh.shape}")
print(f"b_h: {b_h.shape}")

输入形状：
x_t: (2, 3)
h_prev: (2, 4)
W_hx: (4, 3)
W_hh: (4, 4)
b_h: (4,)


In [13]:
h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)

print("\n前向传播结果：")
print(f"h_t shape: {h_t.shape}")
print(f"h_t:\n{h_t}")


前向传播结果：
h_t shape: (2, 4)
h_t:
[[-0.99460443 -0.77409015 -0.9004889  -0.99810568]
 [-0.92208444  0.97361461  0.99985849 -0.98411114]]


In [14]:
dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

print("\n反向传播结果：")
print(f"dx_t shape: {dx_t.shape}")
print(f"dh_prev shape: {dh_prev.shape}")
print(f"dW_hx shape: {dW_hx.shape}")
print(f"dW_hh shape: {dW_hh.shape}")
print(f"db_h shape: {db_h.shape}")


反向传播结果：
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (4,)


### 梯度验证（数值梯度对比）

使用数值梯度验证反向传播实现的正确性。

In [15]:
def numerical_gradient(f, x, h=1e-5):
    """计算数值梯度"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]
        x[idx] = old_val + h
        f_x_plus = f()
        x[idx] = old_val - h
        f_x_minus = f()
        grad[idx] = (f_x_plus - f_x_minus) / (2 * h)
        x[idx] = old_val
        it.iternext()
    return grad

In [16]:
np.random.seed(42)

batch_size = 1
input_size = 2
hidden_size = 3

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size,)

h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)

dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

def loss_fn():
    h, _ = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    return np.sum(h * dh_next)

dx_t_num = numerical_gradient(loss_fn, x_t)
dh_prev_num = numerical_gradient(loss_fn, h_prev)
dW_hx_num = numerical_gradient(loss_fn, W_hx)
dW_hh_num = numerical_gradient(loss_fn, W_hh)
db_h_num = numerical_gradient(loss_fn, b_h)

print("梯度验证结果：")
print(f"dx_t 误差: {np.mean(np.abs(dx_t - dx_t_num)):.6f}")
print(f"dh_prev 误差: {np.mean(np.abs(dh_prev - dh_prev_num)):.6f}")
print(f"dW_hx 误差: {np.mean(np.abs(dW_hx - dW_hx_num)):.6f}")
print(f"dW_hh 误差: {np.mean(np.abs(dW_hh - dW_hh_num)):.6f}")
print(f"db_h 误差: {np.mean(np.abs(db_h - db_h_num)):.6f}")

print("\n所有梯度验证通过！" if all([
    np.mean(np.abs(dx_t - dx_t_num)) < 1e-3,
    np.mean(np.abs(dh_prev - dh_prev_num)) < 1e-3,
    np.mean(np.abs(dW_hx - dW_hx_num)) < 1e-3,
    np.mean(np.abs(dW_hh - dW_hh_num)) < 1e-3,
    np.mean(np.abs(db_h - db_h_num)) < 1e-3
]) else "梯度验证失败！")

梯度验证结果：
dx_t 误差: 0.000000
dh_prev 误差: 0.000000
dW_hx 误差: 0.000000
dW_hh 误差: 0.000000
db_h 误差: 0.000000

所有梯度验证通过！


## 4.1 理论计算题

假设一个深度双向 RNN，有 L 层，每层隐藏单元数为 H，输入维度为 D，输出维度为 O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影。

### 参数计算过程

**1. 单方向单层 RNN 的参数**

一个标准的 RNN 层包含：
- $W_{hx}$：输入到隐藏的权重，形状 (H, input_dim)
- $W_{hh}$：隐藏到隐藏的权重，形状 (H, H)
- $b_h$：隐藏层偏置，形状 (H,)

参数总数：$H \times input\_dim + H \times H + H = H(input\_dim + H + 1)$

**2. 双向单层 RNN 的参数**

双向 RNN 包含前向和后向两个 RNN：
- 前向 RNN：$H(input\_dim + H + 1)$
- 后向 RNN：$H(input\_dim + H + 1)$

参数总数：$2H(input\_dim + H + 1)$

**3. 深度双向 RNN 的参数**

第 1 层输入维度为 D，后续层输入维度为 2H（前向和后向拼接）：
- 第 1 层：$2H(D + H + 1)$
- 第 2 到 L 层（共 L-1 层）：每层 $2H(2H + H + 1) = 2H(3H + 1)$

**4. 输出层参数**

输出层将最后一层的拼接隐藏状态（维度 2H）映射到输出维度 O：
- $W_{oh}$：形状 (O, 2H)
- $b_o$：形状 (O,)

参数总数：$O \times 2H + O = O(2H + 1)$

### 参数总数表达式

综合以上分析，深度双向 RNN 的参数总数为：

$$
\text{Total} = 2H(D + H + 1) + 2H(3H + 1)(L - 1) + O(2H + 1)
$$

**化简后：**

$$
\text{Total} = 2H\left[(3H + 1)L + D - 2H\right] + O(2H + 1)
$$

**验证：当 L = 1（单层双向 RNN）时：**

$$
\text{Total} = 2H(D + H + 1) + O(2H + 1)
$$

符合预期。

## 4.2 编程题



In [17]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向 RNN 编码器
    
    参数：
    input_dim: 输入维度
    hidden_dim: 单向隐藏层维度
    """
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        
        self.forward_rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=False,
            bidirectional=False
        )
        
        self.backward_rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=False,
            bidirectional=False
        )

    def forward(self, X):
        """
        前向传播
        
        参数：
        X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
        outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_hidden: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        forward_out, _ = self.forward_rnn(X)
        
        X_reversed = X.flip(0)
        backward_out, _ = self.backward_rnn(X_reversed)
        backward_out = backward_out.flip(0)
        
        outputs = torch.cat([forward_out, backward_out], dim=-1)
        
        final_hidden = torch.cat(
            [forward_out[-1], backward_out[-1]],
            dim=-1
        )
        
        return outputs, final_hidden

### 使用 torch.nn.RNN 的双向模式实现（更简洁）

In [18]:
class BidirectionalRNNEncoderBuiltin(nn.Module):
    """
    使用 PyTorch 内置双向 RNN 的编码器
    """
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoderBuiltin, self).__init__()
        self.hidden_dim = hidden_dim
        
        self.bi_rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=False,
            bidirectional=True
        )

    def forward(self, X):
        """
        前向传播
        
        参数：
        X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
        outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_hidden: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        outputs, hidden = self.bi_rnn(X)
        
        final_hidden = torch.cat([hidden[0], hidden[1]], dim=-1)
        
        return outputs, final_hidden

### 测试示例

In [19]:
torch.manual_seed(42)

seq_len = 5
batch_size = 2
input_dim = 3
hidden_dim = 4

X = torch.randn(seq_len, batch_size, input_dim)

print("输入形状：")
print(f"X: {X.shape}")
print(f"seq_len = {seq_len}")
print(f"batch_size = {batch_size}")
print(f"input_dim = {input_dim}")
print(f"hidden_dim = {hidden_dim}")

输入形状：
X: torch.Size([5, 2, 3])
seq_len = 5
batch_size = 2
input_dim = 3
hidden_dim = 4


In [20]:
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)

outputs, final_hidden = encoder(X)

print("\n手动实现的双向 RNN 编码器：")
print(f"outputs shape: {outputs.shape}")
print(f"final_hidden shape: {final_hidden.shape}")

print("\n验证输出形状：")
print(f"outputs 期望形状: ({seq_len}, {batch_size}, {2*hidden_dim})")
print(f"final_hidden 期望形状: ({batch_size}, {2*hidden_dim})")
print(f"形状正确: {outputs.shape == (seq_len, batch_size, 2*hidden_dim) and final_hidden.shape == (batch_size, 2*hidden_dim)}")


手动实现的双向 RNN 编码器：
outputs shape: torch.Size([5, 2, 8])
final_hidden shape: torch.Size([2, 8])

验证输出形状：
outputs 期望形状: (5, 2, 8)
final_hidden 期望形状: (2, 8)
形状正确: True


In [21]:
encoder_builtin = BidirectionalRNNEncoderBuiltin(input_dim, hidden_dim)

outputs_builtin, final_hidden_builtin = encoder_builtin(X)

print("\n内置双向 RNN 编码器：")
print(f"outputs shape: {outputs_builtin.shape}")
print(f"final_hidden shape: {final_hidden_builtin.shape}")

print("\n验证输出形状：")
print(f"形状正确: {outputs_builtin.shape == (seq_len, batch_size, 2*hidden_dim) and final_hidden_builtin.shape == (batch_size, 2*hidden_dim)}")


内置双向 RNN 编码器：
outputs shape: torch.Size([5, 2, 8])
final_hidden shape: torch.Size([2, 8])

验证输出形状：
形状正确: True


## 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $\mathbf{v}_c, \mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$，写出完整的目标函数。

### Skip-gram 模型基本思想

Skip-gram 模型的目标是给定中心词 $w_c$，预测其上下文词 $w_o$。模型通过最大化真实上下文词的概率，同时最小化负样本（噪声词）的概率来训练词向量。

对于正样本 $(w_c, w_o)$，模型预测为正类的概率（使用 sigmoid 函数）：

$$P(D=1|w_c, w_o) = \sigma(\mathbf{u}_o^T \mathbf{v}_c)$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 是 sigmoid 函数。

对于负样本 $(w_c, w_{n_k})$，模型预测为负类的概率：

$$P(D=0|w_c, w_{n_k}) = 1 - \sigma(\mathbf{u}_{n_k}^T \mathbf{v}_c) = \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$$

### 负采样的目标函数

负采样的核心思想是：对于每个正样本 $(w_c, w_o)$，采样 $K$ 个负样本 $(w_c, w_{n_1}), (w_c, w_{n_2}), ..., (w_c, w_{n_K})$，然后最大化正样本为正类的概率和所有负样本为负类的概率。

**对数似然函数（损失函数的相反数）：**

$$
\mathcal{L} = \log P(D=1|w_c, w_o) + \sum_{k=1}^K \log P(D=0|w_c, w_{n_k})
$$

代入概率表达式：

$$
\mathcal{L} = \log \sigma(\mathbf{u}_o^T \mathbf{v}_c) + \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)
$$

**损失函数（最小化目标）：**

$$
J = -\mathcal{L} = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)
$$

### 负样本采样方法

负样本从噪声分布 $P_n(w)$ 中采样，通常使用幂次分布：

$$
P_n(w) = \frac{freq(w)^{3/4}}{\sum_{w'} freq(w')^{3/4}}
$$

其中 $freq(w)$ 是词 $w$ 在语料中的出现频率。使用 $3/4$ 次幂的原因是：

1. 降低高频词被采样的概率，避免高频词（如 "the", "a"）主导负样本
2. 增加低频词被采样的概率，使得低频词也能得到充分训练

**采样过程**：
1. 对语料中每个词，计算其 $freq(w)^{3/4}$
2. 归一化得到概率分布 $P_n(w)$
3. 根据 $P_n(w)$ 采样 $K$ 个负样本（注意要排除正样本本身）

### 完整目标函数

综合以上分析，Skip-gram 负采样的完整目标函数为：

$$
\min_{\mathbf{v}, \mathbf{u}} -\frac{1}{T} \sum_{t=1}^T \sum_{-m \leq j \leq m, j \neq 0} \
\left[ \log \sigma(\mathbf{u}_{w_{t+j}}^T \mathbf{v}_{w_t}) + \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_{w_t}) \right]
$$

其中：
- $T$ 是语料中的词数
- $m$ 是上下文窗口大小
- $\mathbf{v}_w$ 是词 $w$ 的输入向量（中心词向量）
- $\mathbf{u}_w$ 是词 $w$ 的输出向量（上下文词向量）
- $n_k$ 是从噪声分布 $P_n$ 中采样的第 $k$ 个负样本

## 5.2 编程题



In [22]:
import numpy as np

def softmax(x):
    """计算 softmax 函数"""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def cbow_forward(context_indices, center_indices, W, W_out):
    """
    CBOW 模型前向传播和损失计算
    
    参数：
    context_indices: 上下文词索引列表，形状 (batch_size, context_size)
    center_indices: 中心词索引列表，形状 (batch_size,)
    W: 输入权重矩阵（词嵌入矩阵），形状 (V, d)
    W_out: 输出权重矩阵，形状 (d, V)
    
    返回：
    loss: 交叉熵损失值
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    context_vectors = W[context_indices]
    
    hidden = np.mean(context_vectors, axis=1)
    
    logits = np.dot(hidden, W_out)
    
    probs = softmax(logits)
    
    log_probs = -np.log(probs[np.arange(batch_size), center_indices])
    loss = np.mean(log_probs)
    
    return loss

### 测试示例

In [23]:
np.random.seed(42)

V = 10
d = 5
batch_size = 3
context_size = 4

W = np.random.randn(V, d)
W_out = np.random.randn(d, V)

context_indices = np.random.randint(0, V, size=(batch_size, context_size))
center_indices = np.random.randint(0, V, size=batch_size)

print("输入信息：")
print(f"词汇表大小 V: {V}")
print(f"嵌入维度 d: {d}")
print(f"批次大小 batch_size: {batch_size}")
print(f"上下文大小 context_size: {context_size}")
print(f"上下文索引:\n{context_indices}")
print(f"中心词索引: {center_indices}")

输入信息：
词汇表大小 V: 10
嵌入维度 d: 5
批次大小 batch_size: 3
上下文大小 context_size: 4
上下文索引:
[[8 4 0 2]
 [9 7 5 7]
 [8 3 0 0]]
中心词索引: [9 3 6]


In [24]:
loss = cbow_forward(context_indices, center_indices, W, W_out)

print(f"\nCBOW 损失值: {loss:.6f}")


CBOW 损失值: 2.888937


### 使用 PyTorch 验证

In [25]:
import torch
import torch.nn as nn

torch.manual_seed(42)

W_torch = torch.tensor(W, dtype=torch.float32, requires_grad=True)
W_out_torch = torch.tensor(W_out, dtype=torch.float32, requires_grad=True)

context_indices_torch = torch.tensor(context_indices, dtype=torch.long)
center_indices_torch = torch.tensor(center_indices, dtype=torch.long)

context_vectors = W_torch[context_indices_torch]
hidden = context_vectors.mean(dim=1)
logits = hidden @ W_out_torch

criterion = nn.CrossEntropyLoss()
loss_torch = criterion(logits, center_indices_torch)

print(f"NumPy 实现损失: {loss:.6f}")
print(f"PyTorch 实现损失: {loss_torch.item():.6f}")
print(f"两者一致: {np.abs(loss - loss_torch.item()) < 1e-6}")

NumPy 实现损失: 2.888937
PyTorch 实现损失: 2.888937
两者一致: True


## 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵。

使用 $score = QK^T / \sqrt{d_k}$，其中 $d_k = 4$。

要求写出中间步骤：
1. 计算得分矩阵
2. 对得分矩阵做 softmax
3. 加权求和得到输出

In [26]:
import numpy as np

np.random.seed(42)

Q = np.random.randn(2, 4)
K = np.random.randn(3, 4)
V = np.random.randn(3, 5)

d_k = Q.shape[1]

print("输入矩阵：")
print(f"Q ({Q.shape}):")
print(Q)
print(f"\nK ({K.shape}):")
print(K)
print(f"\nV ({V.shape}):")
print(V)
print(f"\nd_k = {d_k}")

输入矩阵：
Q ((2, 4)):
[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [-0.23415337 -0.23413696  1.57921282  0.76743473]]

K ((3, 4)):
[[-0.46947439  0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]
 [-1.01283112  0.31424733 -0.90802408 -1.4123037 ]]

V ((3, 5)):
[[ 1.46564877 -0.2257763   0.0675282  -1.42474819 -0.54438272]
 [ 0.11092259 -1.15099358  0.37569802 -0.60063869 -0.29169375]
 [-0.60170661  1.85227818 -0.01349722 -1.05771093  0.82254491]]

d_k = 4


In [27]:
scores = np.dot(Q, K.T) / np.sqrt(d_k)

print("步骤 1：计算得分矩阵")
print(f"QK^T ({scores.shape}):")
print(np.dot(Q, K.T))
print(f"\nscore = QK^T / sqrt(d_k) ({scores.shape}):")
print(scores)

步骤 1：计算得分矩阵
QK^T ((2, 3)):
[[-1.3176819  -1.58886576 -3.28563423]
 [-1.10635669 -2.76421799 -2.35423325]]

score = QK^T / sqrt(d_k) ((2, 3)):
[[-0.65884095 -0.79443288 -1.64281711]
 [-0.55317835 -1.382109   -1.17711663]]


In [28]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

attention_weights = softmax(scores)

print("步骤 2：计算注意力权重（softmax）")
print(f"attention_weights ({attention_weights.shape}):")
print(attention_weights)
print(f"\n每行求和验证: {np.sum(attention_weights, axis=-1)}")

步骤 2：计算注意力权重（softmax）
attention_weights ((2, 3)):
[[0.44503374 0.38860296 0.1663633 ]
 [0.50701047 0.22131809 0.27167143]]

每行求和验证: [1. 1.]


In [29]:
output = np.dot(attention_weights, V)

print("步骤 3：加权求和得到输出")
print(f"output ({output.shape}):")
print(output)

步骤 3：加权求和得到输出
output ((2, 5)):
[[ 0.5952661  -0.23960648  0.17380425 -1.04343526 -0.21878045]
 [ 0.60418195  0.13400442  0.11371947 -1.1426443  -0.11710289]]


## 6.2 编程题



In [30]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    
    参数：
    d_model: 模型维度
    num_heads: 头数
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V):
        """
        缩放点积注意力
        
        参数：
        Q: 形状 (batch, num_heads, seq_len, d_k)
        K: 形状 (batch, num_heads, seq_len, d_k)
        V: 形状 (batch, num_heads, seq_len, d_v)
        
        返回：
        output: 形状 (batch, num_heads, seq_len, d_v)
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output

    def split_heads(self, x):
        """
        将输入分割为多个头
        
        参数：
        x: 形状 (seq_len, batch, d_model)
        
        返回：
        x: 形状 (batch, num_heads, seq_len, d_k)
        """
        seq_len, batch, d_model = x.shape
        x = x.transpose(0, 1).contiguous()
        x = x.view(batch, -1, self.num_heads, self.d_k)
        x = x.transpose(1, 2).contiguous()
        return x

    def merge_heads(self, x):
        """
        将多个头的输出合并
        
        参数：
        x: 形状 (batch, num_heads, seq_len, d_k)
        
        返回：
        x: 形状 (seq_len, batch, d_model)
        """
        batch, num_heads, seq_len, d_k = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch, seq_len, -1)
        x = x.transpose(0, 1).contiguous()
        return x

    def forward(self, X):
        """
        前向传播
        
        参数：
        X: 形状 (seq_len, batch, d_model)
        
        返回：
        output: 形状 (seq_len, batch, d_model)
        """
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        attention_output = self.scaled_dot_product_attention(Q, K, V)
        
        output = self.merge_heads(attention_output)
        output = self.W_o(output)
        
        return output

### 测试示例

In [31]:
torch.manual_seed(42)

seq_len = 3
batch_size = 2
d_model = 4
num_heads = 2

X = torch.randn(seq_len, batch_size, d_model)

print("输入信息：")
print(f"X shape: {X.shape}")
print(f"seq_len = {seq_len}")
print(f"batch_size = {batch_size}")
print(f"d_model = {d_model}")
print(f"num_heads = {num_heads}")
print(f"每个头维度 d_k = {d_model // num_heads}")

输入信息：
X shape: torch.Size([3, 2, 4])
seq_len = 3
batch_size = 2
d_model = 4
num_heads = 2
每个头维度 d_k = 2


In [32]:
multi_head_attn = MultiHeadAttention(d_model, num_heads)

output = multi_head_attn(X)

print("\n多头注意力输出：")
print(f"output shape: {output.shape}")
print(f"期望形状: ({seq_len}, {batch_size}, {d_model})")
print(f"形状正确: {output.shape == (seq_len, batch_size, d_model)}")


多头注意力输出：
output shape: torch.Size([3, 2, 4])
期望形状: (3, 2, 4)
形状正确: True


### 使用 PyTorch 官方实现验证

In [33]:
class MultiHeadAttentionBuiltin(nn.Module):
    """
    使用 PyTorch 内置 nn.MultiheadAttention 的实现
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttentionBuiltin, self).__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=False)

    def forward(self, X):
        output, _ = self.attn(X, X, X)
        return output

torch.manual_seed(42)
multi_head_attn_builtin = MultiHeadAttentionBuiltin(d_model, num_heads)

output_builtin = multi_head_attn_builtin(X)

print(f"自定义实现输出形状: {output.shape}")
print(f"内置实现输出形状: {output_builtin.shape}")
print(f"形状一致: {output.shape == output_builtin.shape}")

print(f"\n自定义实现参数数量: {sum(p.numel() for p in multi_head_attn.parameters())}")
print(f"内置实现参数数量: {sum(p.numel() for p in multi_head_attn_builtin.parameters())}")

自定义实现输出形状: torch.Size([3, 2, 4])
内置实现输出形状: torch.Size([3, 2, 4])
形状一致: True

自定义实现参数数量: 80
内置实现参数数量: 80
